# Community Notes 陣営反応分析 — 全データ実行

## 手順
1. このノートブックを Colab で開く
2. 上から順にセルを実行

データは共有 Drive「基礎プロジェクト/data/」から自動でコピーします。

## 0. セットアップ

In [ ]:
# Google Drive をマウント
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ====================================================
# ★ Drive のデータフォルダのパスをここで設定 ★
# 共有ドライブの場合: /content/drive/Shareddrives/...
# マイドライブの場合: /content/drive/MyDrive/...
# ====================================================
DRIVE_DATA = '/content/drive/MyDrive/基礎プロジェクト/data'

# パスが正しいか確認
import os
if os.path.exists(DRIVE_DATA):
    print('OK: フォルダが見つかりました')
    !ls "$DRIVE_DATA"
else:
    print(f'ERROR: {DRIVE_DATA} が見つかりません')
    print('Drive 内のパスを確認して DRIVE_DATA を修正してください')
    print()
    print('--- Drive のルート ---')
    !ls /content/drive/MyDrive/ | head -20
    print()
    print('--- 共有ドライブ ---')
    !ls /content/drive/Shareddrives/ 2>/dev/null || echo '共有ドライブなし'

In [ ]:
# リポジトリをクローン
!git clone https://github.com/hirototoda/toriumi_x3.git /content/toriumi_x3 2>/dev/null || echo 'already cloned'
os.chdir('/content/toriumi_x3')
!git pull
!pwd

In [ ]:
# 依存パッケージ
!pip install -q statsmodels

In [ ]:
# Drive の zip を data/raw/ に解凍
import glob

raw_dir = '/content/toriumi_x3/data/raw'
os.makedirs(raw_dir, exist_ok=True)

# ratings (RatingData/)
for z in sorted(glob.glob(f'{DRIVE_DATA}/RatingData/ratings-*.zip')):
    print(f'Unzipping {os.path.basename(z)} ...')
    !unzip -o -q "$z" -d "$raw_dir"

# notes (Notes data/)
for z in sorted(glob.glob(f'{DRIVE_DATA}/Notes data/notes-*.zip')):
    print(f'Unzipping {os.path.basename(z)} ...')
    !unzip -o -q "$z" -d "$raw_dir"

# noteStatusHistory (Note status history data/)
for z in sorted(glob.glob(f'{DRIVE_DATA}/Note status history data/noteStatusHistory-*.zip')):
    print(f'Unzipping {os.path.basename(z)} ...')
    !unzip -o -q "$z" -d "$raw_dir"

print()
print('=== data/raw/ ===')
!ls -lh data/raw/*.tsv

## 1. 全トピックでパイプライン実行

In [ ]:
!python scripts/run_pipeline.py

## 2. トピック別比較

In [ ]:
!python scripts/run_by_topic.py

## 3. 結果の確認

In [ ]:
import pandas as pd

# トピック別比較表
df = pd.read_csv('data/processed/topic_comparison.csv')
display(df)

In [ ]:
# ターゲットノート
targets = pd.read_csv('data/processed/target_notes.csv')
print(f'ターゲットノート数: {len(targets)}')
display(targets.head(20))

In [ ]:
# バースト一覧
bursts = pd.read_csv('data/processed/bursts.csv')
print(f'バースト数: {len(bursts)}')
print(bursts['burst_type'].value_counts())

## 4. 結果を Drive に保存

In [ ]:
save_dir = f'{DRIVE_DATA}/../results'
os.makedirs(save_dir, exist_ok=True)
!cp data/processed/*.csv "$save_dir"/
!cp data/processed/*.png "$save_dir"/ 2>/dev/null
print(f'Done! Results saved to: {save_dir}')